# Extractor-derived G-code reprint segments

This notebook reproduces the **segment-based reprint map** that uses the **extractor evidence**.

Workflow:
1. Run the extractor across the scan frames and merge its no-filament evidence.
2. Parse the G-code into real extrusion segments.
3. Sample the extractor evidence along each G-code segment.
4. Score each segment.
5. Smooth the scores in G-code order.
6. Highlight segments above a user-chosen threshold.

The expensive part is step 1. After you build the cached evidence once, you can change the threshold and redraw quickly.

In [4]:
import os
import math
import json
import importlib.util
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1) Configuration

Change these paths if needed.

In [5]:
# --- Input files ---
FOLDER = r"C:\Users\dhruv\Documents\dhruv_python\disc2accurate"   # folder of scan images
GCODE_FILE = r"C:\Users\dhruv\Documents\proforge4correction\sensor\P4_one_layer_annular_disc_60OD_12ID_0p20H.gcode"
FILAMENT_ARRAY_PY = r"C:\Users\dhruv\Documents\proforge4correction\sensor\filament_array.py"
GCODE_COORD_MASK_PY = r"C:\Users\dhruv\Documents\proforge4correction\sensor\gcode_coordinate_mask.py"
MERGE_EVIDENCE_PY = r"C:\Users\dhruv\Documents\proforge4correction\sensor\merge_miss_print_coord.py"

# If your zip has not been extracted yet, set this and run the extraction cell below.
# ZIP_FILE = r"/mnt/data/disc2accurate(4).zip"
# EXTRACT_TO = r"/mnt/data/work"

# --- Merge / extractor settings ---
RADIUS = 35
THRESHOLD = 0.15
PX_PER_MM = 26.13
FLIP_X = False
FLIP_Y = True
SWAP_XY = False
LAYER_INDEX = 0
GCODE_LINE_WIDTH_MM = 0.45
MAX_FRAME_Z = 0.20
MARGIN_PX = 200

# --- Segment scoring settings ---
SMOOTH_WINDOW = 21
WEIGHT_MEAN = 0.45
WEIGHT_Q75 = 0.30
WEIGHT_FRAC05 = 0.15
WEIGHT_FRAC10 = 0.10

# --- Visualization threshold ---
# Lower quantile => more red segments.
# Examples:
# 0.70 = top 30%
# 0.60 = top 40%
# 0.50 = top 50%
# 0.40 = top 60%
# 0.30 = top 70%
THRESHOLD_QUANTILE = 0.50

# --- Cache / outputs ---
CACHE_DIR = Path("./reprint_segment_cache")
CACHE_DIR.mkdir(exist_ok=True)

## Optional: extract the image zip

Only run this if your image folder does not already exist.

In [6]:
import zipfile
from pathlib import Path

if not Path(FOLDER).exists():
    Path(EXTRACT_TO).mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_FILE, 'r') as zf:
        zf.extractall(EXTRACT_TO)
    print("Extracted to:", EXTRACT_TO)
else:
    print("Folder already exists:", FOLDER)

Folder already exists: C:\Users\dhruv\Documents\dhruv_python\disc2accurate


## 2) Load the helper modules from file paths

In [7]:
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    import sys
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

# Load in dependency order.
gcode_coordinate_mask = load_module("gcode_coordinate_mask", GCODE_COORD_MASK_PY)
filament_array = load_module("filament_array", FILAMENT_ARRAY_PY)
merge_miss_print_evidence = load_module("merge_miss_print_evidence", MERGE_EVIDENCE_PY)

print("Loaded modules:")
print(" -", gcode_coordinate_mask.__file__)
print(" -", filament_array.__file__)
print(" -", merge_miss_print_evidence.__file__)

Loaded modules:
 - C:\Users\dhruv\Documents\proforge4correction\sensor\gcode_coordinate_mask.py
 - C:\Users\dhruv\Documents\proforge4correction\sensor\filament_array.py
 - C:\Users\dhruv\Documents\proforge4correction\sensor\merge_miss_print_coord.py


## 3) Build and cache the extractor evidence

This is the slow step.

It creates:
- `no_ratio.npy`: per-pixel extractor no-filament evidence ratio
- `expected_filament.png`: expected printed region
- `meta.json`: origin and settings used for the map

In [8]:
CACHE_NO_RATIO = CACHE_DIR / "no_ratio.npy"
CACHE_EXPECTED = CACHE_DIR / "expected_filament.png"
CACHE_META = CACHE_DIR / "meta.json"
CACHE_NO_VOTES = CACHE_DIR / "no_votes.npy"
CACHE_EVAL_VOTES = CACHE_DIR / "evaluated_votes.npy"


def build_evidence_cache(force=False):
    if CACHE_NO_RATIO.exists() and CACHE_EXPECTED.exists() and CACHE_META.exists() and not force:
        print("Using existing cache in", CACHE_DIR)
        return

    out = merge_miss_print_evidence.merge_miss_print_evidence(
        folder=FOLDER,
        gcode_file=GCODE_FILE,
        radius=RADIUS,
        threshold=THRESHOLD,
        px_per_mm=PX_PER_MM,
        margin_px=MARGIN_PX,
        flip_x=FLIP_X,
        flip_y=FLIP_Y,
        swap_xy=SWAP_XY,
        layer_index=LAYER_INDEX,
        gcode_line_width_mm=GCODE_LINE_WIDTH_MM,
        max_frame_z=MAX_FRAME_Z,
        min_no_votes=1,
        min_no_ratio=0.0,
        expected_edge_tolerance_mm=0.0,
        merged_out_path=str(CACHE_DIR / "merged_no_filament.png"),
        miss_out_path=str(CACHE_DIR / "miss_print.png"),
        expected_out_path=str(CACHE_EXPECTED),
        debug_prefix=str(CACHE_DIR / "evid")
    )

    np.save(CACHE_NO_RATIO, out["no_ratio"])
    np.save(CACHE_NO_VOTES, out["no_votes"])
    np.save(CACHE_EVAL_VOTES, out["evaluated_votes"])

    meta = {
        "origin_xy_mm": list(map(float, out["origin_xy_mm"])),
        "frames_used": int(out["frames_used"]),
        "px_per_mm": float(PX_PER_MM),
        "radius": int(RADIUS),
        "threshold": float(THRESHOLD),
        "flip_x": bool(FLIP_X),
        "flip_y": bool(FLIP_Y),
        "swap_xy": bool(SWAP_XY),
        "layer_index": int(LAYER_INDEX),
        "smooth_window": int(SMOOTH_WINDOW),
    }
    with open(CACHE_META, "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved cache to", CACHE_DIR)
    print("frames_used:", out["frames_used"])
    print("origin_xy_mm:", out["origin_xy_mm"])

# Run once. Set force=True to rebuild.
build_evidence_cache(force=False)

AttributeError: module 'merge_miss_print_evidence' has no attribute 'merge_miss_print_evidence'

## 4) Load cached evidence and parse the G-code segments

In [ ]:
with open(CACHE_META, "r") as f:
    meta = json.load(f)

no_ratio = np.load(CACHE_NO_RATIO)
no_votes = np.load(CACHE_NO_VOTES)
evaluated_votes = np.load(CACHE_EVAL_VOTES)
expected_mask = cv2.imread(str(CACHE_EXPECTED), cv2.IMREAD_GRAYSCALE) > 0
origin_x_mm, origin_y_mm = meta["origin_xy_mm"]

segments, unsupported = gcode_coordinate_mask.parse_gcode_extrusion_segments(GCODE_FILE)
selected, target_z, layer_zs = gcode_coordinate_mask.pick_layer_segments_by_index(
    segments,
    layer_index=LAYER_INDEX,
    include_previous_layers=False,
    z_tol=0.08,
)
selected = gcode_coordinate_mask.transform_segments_xy(
    selected,
    flip_x=FLIP_X,
    flip_y=FLIP_Y,
    swap_xy=SWAP_XY,
)

print("no_ratio shape:", no_ratio.shape)
print("selected extrusion segments:", len(selected))
print("target_z:", target_z)
print("unsupported commands seen:", len(unsupported))

## 5) Segment scoring functions

In [ ]:
def sample_segment_pixels(x0, y0, x1, y1, origin_x_mm, origin_y_mm, px_per_mm, shape):
    h, w = shape
    dist_mm = math.hypot(x1 - x0, y1 - y0)
    n = max(2, int(math.ceil(dist_mm * px_per_mm * 1.5)))
    t = np.linspace(0.0, 1.0, n)
    xs = x0 + (x1 - x0) * t
    ys = y0 + (y1 - y0) * t
    cols = np.round((xs - origin_x_mm) * px_per_mm).astype(int)
    rows = np.round((ys - origin_y_mm) * px_per_mm).astype(int)
    good = (rows >= 0) & (rows < h) & (cols >= 0) & (cols < w)
    return rows[good], cols[good]


def score_segments_from_extractor(selected_segments, no_ratio, origin_x_mm, origin_y_mm, px_per_mm):
    scores = []
    h, w = no_ratio.shape

    for idx, seg in enumerate(selected_segments):
        x0, y0, _, x1, y1, _, _, line_no = seg
        rr, cc = sample_segment_pixels(x0, y0, x1, y1, origin_x_mm, origin_y_mm, px_per_mm, (h, w))
        if len(rr) == 0:
            vals = np.array([0.0], dtype=np.float32)
        else:
            vals = no_ratio[rr, cc]

        mean_val = float(vals.mean())
        q75_val = float(np.quantile(vals, 0.75))
        frac05 = float((vals >= 0.05).mean())
        frac10 = float((vals >= 0.10).mean())

        severity = (
            WEIGHT_MEAN * mean_val
            + WEIGHT_Q75 * q75_val
            + WEIGHT_FRAC05 * frac05
            + WEIGHT_FRAC10 * frac10
        )

        scores.append({
            "idx": idx,
            "line_no": int(line_no),
            "x0": float(x0), "y0": float(y0),
            "x1": float(x1), "y1": float(y1),
            "mean": mean_val,
            "q75": q75_val,
            "frac05": frac05,
            "frac10": frac10,
            "severity": severity,
        })

    return scores


def smooth_segment_scores(scores, window=21):
    arr = np.array([s["severity"] for s in scores], dtype=float)
    pad = window // 2
    smoothed = np.convolve(np.pad(arr, (pad, pad), mode="edge"), np.ones(window) / window, mode="valid")
    for s, val in zip(scores, smoothed):
        s["smoothed_severity"] = float(val)
    return scores

## 6) Score the G-code segments

In [ ]:
segment_scores = score_segments_from_extractor(
    selected,
    no_ratio,
    origin_x_mm,
    origin_y_mm,
    PX_PER_MM,
)
segment_scores = smooth_segment_scores(segment_scores, window=SMOOTH_WINDOW)

sm = np.array([s["smoothed_severity"] for s in segment_scores])
print("smoothed severity stats")
print("  min   =", sm.min())
print("  q25   =", np.quantile(sm, 0.25))
print("  q50   =", np.quantile(sm, 0.50))
print("  q60   =", np.quantile(sm, 0.60))
print("  q70   =", np.quantile(sm, 0.70))
print("  q80   =", np.quantile(sm, 0.80))
print("  max   =", sm.max())

## 7) Draw the thresholded reprint map

Change `THRESHOLD_QUANTILE` in the config cell, rerun the config cell if needed, then rerun this cell.

In [ ]:
def render_thresholded_segment_map(segment_scores, expected_mask, threshold_quantile=0.50, px_per_mm=26.13,
                                   origin_x_mm=0.0, origin_y_mm=0.0, line_thickness=2,
                                   save_path=None, title=None):
    threshold_value = float(np.quantile([s["smoothed_severity"] for s in segment_scores], threshold_quantile))

    base = np.zeros((*expected_mask.shape, 3), dtype=np.uint8)
    base[expected_mask] = (210, 210, 210)

    highlighted = 0
    for s in segment_scores:
        if s["smoothed_severity"] < threshold_value:
            continue
        c0 = int(round((s["x0"] - origin_x_mm) * px_per_mm))
        r0 = int(round((s["y0"] - origin_y_mm) * px_per_mm))
        c1 = int(round((s["x1"] - origin_x_mm) * px_per_mm))
        r1 = int(round((s["y1"] - origin_y_mm) * px_per_mm))
        cv2.line(base, (c0, r0), (c1, r1), (30, 30, 255), line_thickness, cv2.LINE_AA)
        highlighted += 1

    if save_path is not None:
        cv2.imwrite(str(save_path), base)

    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(base, cv2.COLOR_BGR2RGB))
    if title is None:
        title = f"Highlighted segments: top {int((1 - threshold_quantile) * 100)}% by smoothed extractor severity"
    plt.title(title)
    plt.axis("off")
    plt.show()

    print("threshold_quantile:", threshold_quantile)
    print("threshold_value   :", threshold_value)
    print("segments drawn    :", highlighted, "/", len(segment_scores))
    return base, threshold_value

rendered_img, threshold_value = render_thresholded_segment_map(
    segment_scores,
    expected_mask,
    threshold_quantile=THRESHOLD_QUANTILE,
    px_per_mm=PX_PER_MM,
    origin_x_mm=origin_x_mm,
    origin_y_mm=origin_y_mm,
    line_thickness=2,
    save_path=CACHE_DIR / f"reprint_segments_q{int(THRESHOLD_QUANTILE * 100):02d}.png",
)

## 8) Quick comparison across several thresholds

In [ ]:
compare_quantiles = [0.70, 0.60, 0.50, 0.40, 0.30]

fig, axes = plt.subplots(1, len(compare_quantiles), figsize=(5 * len(compare_quantiles), 5))
if len(compare_quantiles) == 1:
    axes = [axes]

for ax, q in zip(axes, compare_quantiles):
    threshold_value = float(np.quantile([s["smoothed_severity"] for s in segment_scores], q))
    img = np.zeros((*expected_mask.shape, 3), dtype=np.uint8)
    img[expected_mask] = (210, 210, 210)

    count = 0
    for s in segment_scores:
        if s["smoothed_severity"] < threshold_value:
            continue
        c0 = int(round((s["x0"] - origin_x_mm) * PX_PER_MM))
        r0 = int(round((s["y0"] - origin_y_mm) * PX_PER_MM))
        c1 = int(round((s["x1"] - origin_x_mm) * PX_PER_MM))
        r1 = int(round((s["y1"] - origin_y_mm) * PX_PER_MM))
        cv2.line(img, (c0, r0), (c1, r1), (30, 30, 255), 2, cv2.LINE_AA)
        count += 1

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"q={q:.2f}
count={count}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 9) Optional: export a table of the most suspicious segments

In [ ]:
import pandas as pd

df = pd.DataFrame(segment_scores)
df = df.sort_values("smoothed_severity", ascending=False)
df.head(20)

## Notes

- This notebook uses the **extractor evidence** as requested.
- The threshold only changes **which G-code segments are highlighted**.
- Lowering `THRESHOLD_QUANTILE` will highlight more of the wall rings.
- If you want the rings to wrap around more continuously, try:
  - `THRESHOLD_QUANTILE = 0.40`
  - `THRESHOLD_QUANTILE = 0.30`
- If you want more continuity, you can also increase `SMOOTH_WINDOW` a little.